# Exploration of various ways to identify the reaches from the distance-to-kinect data

In [ ]:
### imports and globals 

import pyxdf

# for the tests
import numpy as np
import matplotlib.pyplot as plt

from scipy.signal import butter, filtfilt, find_peaks


%matplotlib qt

# optional visualizations within functions
do_visualize = True

## Functions to manage rearm xdf files 

In [ ]:
### xdf functions


def print_streams_types_and_names(fullFname_or_streams):
    """Print the names and types of all streams in the xdf file or in the streams list"""

    if isinstance(fullFname_or_streams, str):
        xdf_data, header = pyxdf.load_xdf(filename=fullFname_or_streams, verbose=False)
    elif (
        isinstance(fullFname_or_streams, list)
        and all(isinstance(x, dict) for x in fullFname_or_streams)
        and all("info" in x for x in fullFname_or_streams)
    ):
        xdf_data = fullFname_or_streams
    else:
        raise ValueError("The first argument must be a filename or a list of streams")

    for i in range(len(xdf_data)):
        stream = xdf_data[i]
        s_type = stream["info"]["type"][0]
        s_name = stream["info"]["name"][0]
        print(f"Stream {i}: {s_type}, {s_name}")


def get_stream(xdf_data, searched_stream_type, searched_stream_names):
    """Get the stream of type 'searched_stream_type' with name in 'searched_stream_names' in the xdf_data"""

    if not isinstance(
        searched_stream_names, list
    ):  # if we get a string (only one name)
        searched_stream_names = [searched_stream_names]

    found_streams = []
    for stream in xdf_data:
        stream_type = stream["info"]["type"][0]
        if searched_stream_type == stream_type:
            stream_name = stream["info"]["name"][0]
            for searched_stream_name in searched_stream_names:
                if searched_stream_name == stream_name:
                    found_streams.append(stream)

    if not found_streams:
        # msg = f" Stream not found. Searched in [{searched_stream_type}:{searched_stream_names}]."
        # print(msg)
        return None

    if len(found_streams) > 1:
        found_streams_names = [stream["info"]["name"][0] for stream in found_streams]
        msg = f"Found multiple streams: [{searched_stream_type},{found_streams_names}]."
        raise ValueError(msg)

    return found_streams[0]


def get_kinect_channel(kinect_mocap, channel_name):
    """Get one channel from the kinect mocap by its name"""
    channel_index = -1
    nb_channels = len(kinect_mocap["info"]["desc"][0]["channels"][0]["channel"])
    for i in range(nb_channels):
        current_name = kinect_mocap["info"]["desc"][0]["channels"][0]["channel"][i][
            "label"
        ][0]
        if current_name == channel_name:
            channel_index = i
            break
    if channel_index == -1:
        raise ValueError(f"Joint {channel_name} not found in the kinect mocap data")

    channel_data = kinect_mocap["time_series"][:, channel_index]

    return channel_data


def interpolate_to_constant_time_step(t, x, dt=0.033):
    """Interpolate the data to a constant time step"""

    n_columns = x.shape[1] if x.ndim > 1 else 1

    t_new = np.arange(t[0], t[-1], dt)

    if x.ndim < 2:
        x_new = np.interp(t_new, t, x)
    else:
        x_new = np.zeros((len(t_new), n_columns))
        for i in range(n_columns):
            x_new[:, i] = np.interp(t_new, t, x[:, i])

    return x_new, t_new

## Load the xdf file

In [ ]:
xdf_fullFname = "../dat/ReArm.lnk/ReArm_C1P07/ReArm_C1P07_20211116_V3/ReArm_C1P02_20210715_V3_Reaching/ReArm_C1P07_20211116_V3_r.xdf"
xdf_fullFname = "../dat/ReArm.lnk/DATA_Named/C1P38/V1/Reaching/task-V1_Reach.xdf"  # no mouse data --> eventIDE
xdf_fullFname = "../dat/ReArm.lnk/DATA_Named/C1P38/V2/Reaching/task-V2_Reach.xdf"  # no mouse data --> eventIDE
xdf_fullFname = "../dat/ReArm.lnk/DATA_named/C1P04/V1/Reaching/C01P04_QueAla_20210531_1_r.xdf"  # old data
xdf_fullFname = "../dat/ReArm.lnk/DATA_named/C1P01/V1/Reaching/001_BenMus_20210202_1_r.xdf"  # large block of zeros TODO
# xdf_fullFname = "../dat/ReArm.lnk/DATA_Named/C1P45/V2/Reaching/task-V2_Reach.xdf"  # no mouse data --> eventIDE
# xdf_fullFname = "../dat/ReArm.lnk/DATA_Named/C1P31/V2/Reaching/C1P31_RauMic_20230331_2_r.xdf"  # wrong correction? look good
xdf_fullFname = "../dat/ReArm.lnk/DATA_Named/C1P31/V3/Reaching/task-V3_Reach.xdf"  # no mouse data --> eventIDE
# xdf_fullFname = "../dat/ReArm.lnk/Fichiers de Karima Bakhti - C1P42/V1/Reaching/ReArm_C1P42_20240603_V1_r.xdf"  # Kinect markers with 1 value that is empty

xdf_data, header = pyxdf.load_xdf(
    filename=xdf_fullFname,
    select_streams=[
        {"type": "MoCap"},
        {"type": "Markers"},
    ],
    synchronize_clocks=True,
    dejitter_timestamps=False,  # to get the raw timestamps to compare with the CSV
    verbose=False,
)

# TODO : we should locate EARLY (just after loading?) the interpolation of the mocap data
# because kinect and mouse are NOT a continuous stream (holes in the data)

print_streams_types_and_names(xdf_data)

### Remove the kinect samples filled with zeros
This has to be done before any processing of the kinect data, as this is to fix a bug due to the kinect. 

In [ ]:
kinect_mocap = get_stream(xdf_data, "MoCap", "EuroMov-Mocap-Kinect")
mouse_mocap = get_stream(xdf_data, "MoCap", "Mouse")

if kinect_mocap:
    kinect_t = kinect_mocap["time_stamps"]
    kinect_data = kinect_mocap["time_series"]
    # find the indexes of kinect data that are filled with zeros
    zero_rows = np.all(kinect_data == 0, axis=1)
    zero_rows_indices = np.where(zero_rows)[0]
    print(f"Found {len(zero_rows_indices)} rows filled with only zeros")
    if len(zero_rows_indices) > 0:
        # remove the zero rows from the data
        kinect_data = np.delete(kinect_data, zero_rows_indices, axis=0)
        kinect_t = np.delete(kinect_t, zero_rows_indices, axis=0)

        WristRight_Z_before = get_kinect_channel(kinect_mocap, "WristRight_Z")
        kinect_t_before = kinect_mocap["time_stamps"]

        # modify the original data
        kinect_mocap["time_series"] = kinect_data
        kinect_mocap["time_stamps"] = kinect_t

        WristRight_Z = get_kinect_channel(kinect_mocap, "WristRight_Z")

        # plot WristRight_Z
        plt.figure()
        plt.plot(kinect_t_before, WristRight_Z_before, ".", label="before")
        plt.plot(kinect_t, WristRight_Z, ".", label="after")
        plt.title("WristRight_Z: before and after removing zero rows")
        plt.xlabel("Time (s)")
        plt.ylabel("Position (m)")
        plt.legend()
        plt.show()

## Interpolate the Mocap data
This is mandatory because the kinect and mouse data are produced by the computer: the sampling rate is not waranted to be constant (samples are forgetten... sometimes). 

In [ ]:
## proceed step by step with visualization
if kinect_mocap:
    # # interpolate all the kinect data to a constant time step

    WristRight_X = get_kinect_channel(kinect_mocap, "WristRight_X")
    WristRight_Y = get_kinect_channel(kinect_mocap, "WristRight_Y")
    WristRight_Z = get_kinect_channel(kinect_mocap, "WristRight_Z")

    WristLeft_X = get_kinect_channel(kinect_mocap, "WristLeft_X")
    WristLeft_Y = get_kinect_channel(kinect_mocap, "WristLeft_Y")
    WristLeft_Z = get_kinect_channel(kinect_mocap, "WristLeft_Z")

    WristLeft_Norm = np.sqrt(WristLeft_X**2 + WristLeft_Y**2 + WristLeft_Z**2)
    WristRight_Norm = np.sqrt(WristRight_X**2 + WristRight_Y**2 + WristRight_Z**2)

    reach_left, reach_t = interpolate_to_constant_time_step(kinect_t, WristLeft_Norm)
    reach_right, reach_t = interpolate_to_constant_time_step(kinect_t, WristRight_Norm)

    # Test : interpolate 2 columns at a time
    w_lr = np.vstack((WristLeft_Norm, WristRight_Norm)).T
    w_lr_, reach_t_ = interpolate_to_constant_time_step(kinect_t, w_lr)
    reach_left_ = w_lr_[:, 0]
    reach_right_ = w_lr_[:, 1]
    # assert that the result is the same
    assert reach_left.shape == reach_left_.shape
    assert reach_right.shape == reach_right_.shape
    assert reach_t.shape == reach_t_.shape
    assert len(reach_left) == len(reach_right)
    assert len(reach_left) == len(reach_t)
    assert len(reach_right) == len(reach_t)
    assert np.allclose(reach_t, reach_t_)
    assert np.allclose(reach_left, reach_left_)
    assert np.allclose(reach_right, reach_right_)
    print("Interpolation multiple column test passed")

    # plot the raw and interpolated data
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(kinect_t, WristLeft_Norm, ".", label="Left wrist", color="b")
    ax.plot(kinect_t, WristRight_Norm, ".", label="Right wrist", color="k")
    ax.plot(reach_t, reach_left, label="Left wrist interpolated", color="b", alpha=0.2)
    ax.plot(
        reach_t, reach_right, label="Right wrist interpolated", color="k", alpha=0.2
    )
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Distance from the kinect (m)")
    ax.legend()
    plt.show()


## real job is below...
def resample_stream(stream):
    """Resample the stream to a constant time step"""
    # get the type and name of the stream
    stream_type = stream["info"]["type"][0]
    stream_name = stream["info"]["name"][0]
    if stream_type != "MoCap":
        return stream

    t = stream["time_stamps"]
    data = stream["time_series"]
    # resample the data to a constant time step
    data, t = interpolate_to_constant_time_step(t, data)
    # modify the original data
    stream["time_series"] = data
    stream["time_stamps"] = t
    return stream


if mouse_mocap:
    mouse_mocap = resample_stream(mouse_mocap)

if kinect_mocap:
    kinect_mocap = resample_stream(kinect_mocap)

## Make the time correction (if needed)

In [ ]:
def get_time_correction(xdf_fullFname):
    """Get the time correction from the xdf file name"""
    time_correction_file = xdf_fullFname.replace(".xdf", "_xdf_time_correction.csv")
    try:
        time_correction = np.loadtxt(time_correction_file, delimiter=",", skiprows=1)
    except FileNotFoundError:
        time_correction = np.float64(0)
    return time_correction


time_correction = get_time_correction(xdf_fullFname)
print(f"Time correction: {time_correction} s")

# make the time correction
kinect_mocap = get_stream(xdf_data, "MoCap", "EuroMov-Mocap-Kinect")
kinect_markers = get_stream(xdf_data, "Markers", "EuroMov-Markers-Kinect")

if kinect_mocap:
    kinect_mocap["time_stamps"] = kinect_mocap["time_stamps"] + time_correction
if kinect_markers:
    kinect_markers["time_stamps"] = kinect_markers["time_stamps"] + time_correction

## Get the markers 

The markers are necessary to identify the *zones of interest* within which the reaches are expected to occur. 
Outside these zones, the data is not relevant for the analysis, and it adds a lot of noise to the data.

Each zone has a *start* and *end* marker, but the label of the markers differs if the sequence was generated by : 

- the software **LSL-Mouse**: stream `mouse_to_nic_markers`
    - start = `"[111]"`  
    - stop = `"[100]"` 

- the software **event-IDE**: stream `event_to_nic_markers`
    - start = `"[100]"`, but we have to keep only the first start in the sequence before each stop
    - stop = `"[75]"`


The streams are loaded from the xdf file using the function `get_stream`:
``` python
# get the markers streams from the xdf file
mouse_to_nic_markers = get_stream(xdf_data, "Markers", ["MouseToNIC"])
event_to_nic_markers = get_stream(xdf_data, "Markers", ["event_ide_TONIC"]) 
```
``` python

### Function to get the markers of the reach zones in the data

In [ ]:
def find_marker_indexes(marker_name, markers_data):
    """Find the indexes of marker_name in markers_data"""

    marker_index_list = [
        i for i, marker in enumerate(markers_data) if marker_name in marker[0]
    ]

    return np.array(marker_index_list)


def get_coherent_start_stop_times(starts, stops):
    """Get the coherent start and stop times from the start and stop times"""

    # make an array of start, 0 and stop, 1
    start = np.zeros(
        len(starts),
        dtype=[("time", float), ("type", int)],
    )
    start["time"] = starts
    start["type"] = 0
    stop = np.zeros(
        len(stops),
        dtype=[("time", float), ("type", int)],
    )
    stop["time"] = stops
    stop["type"] = 1
    start_and_stop = np.concatenate((start, stop))
    start_and_stop = np.sort(start_and_stop, order="time")

    for i in range(len(start_and_stop) - 1):
        # keep only the last start in case of multiple contiguous start
        if start_and_stop[i]["type"] == 0 and start_and_stop[i + 1]["type"] == 0:
            start_and_stop[i + 1]["type"] = -1
        # keep only the first stop in case of multiple contiguousstop
        if start_and_stop[i]["type"] == 1 and start_and_stop[i + 1]["type"] == 1:
            start_and_stop[i + 1]["type"] = -1

    # ensure that the first is a start and the last is a stop
    if start_and_stop[0]["type"] == 1:
        start_and_stop[0]["type"] = -1
    if start_and_stop[-1]["type"] == 0:
        start_and_stop[-1]["type"] = -1

    # clean the start_and_stop array
    start_and_stop_ok = start_and_stop[start_and_stop["type"] != -1]

    starts_corrected = start_and_stop_ok["time"][start_and_stop_ok["type"] == 0]
    stops_corrected = start_and_stop_ok["time"][start_and_stop_ok["type"] == 1]

    if len(starts_corrected) != len(stops_corrected):
        raise ValueError(
            f"Number of starts ({len(starts_corrected)}) and stops ({len(stops_corrected)}) are not equal"
        )

    return starts_corrected, stops_corrected


def get_start_stop_times_from_mouse_to_nic_markers(mouse_to_nic_markers):
    """get the start and stop times from mouse_to_nic_markers"""

    mouse_to_nic_markers_data = mouse_to_nic_markers["time_series"]
    mouse_to_nic_markers_time = mouse_to_nic_markers["time_stamps"]

    if not isinstance(mouse_to_nic_markers_data[0], list):
        mouse_to_nic_markers_data = [[str(x)] for x in mouse_to_nic_markers_data]

    start_marker_index_list = find_marker_indexes("[111]", mouse_to_nic_markers_data)
    stop_marker_index_list = find_marker_indexes("[100]", mouse_to_nic_markers_data)
    start_times_from_mouse_to_nic_markers = mouse_to_nic_markers_time[
        start_marker_index_list
    ]
    stop_times_from_mouse_to_nic_markers = mouse_to_nic_markers_time[
        stop_marker_index_list
    ]

    start_times_from_mouse_to_nic_markers, stop_times_from_mouse_to_nic_markers = (
        get_coherent_start_stop_times(
            start_times_from_mouse_to_nic_markers,
            stop_times_from_mouse_to_nic_markers,
        )
    )

    return (
        start_times_from_mouse_to_nic_markers,
        stop_times_from_mouse_to_nic_markers,
    )


def get_start_stop_times_from_event_ide_TONIC(event_ide_tonic):
    """get the start and stop times from event_ide_tonic"""

    event_ide_markers_data = event_ide_tonic["time_series"]
    event_ide_markers_time = event_ide_tonic["time_stamps"]

    if not isinstance(event_ide_markers_data[0], list):
        event_ide_markers_data = [[str(x)] for x in event_ide_markers_data]

    start_marker_index_list = find_marker_indexes("[100]", event_ide_markers_data)
    stop_marker_index_list = find_marker_indexes("[75]", event_ide_markers_data)

    start_times_from_event_ide_tonic = event_ide_markers_time[start_marker_index_list]
    stop_times_from_event_ide_tonic = event_ide_markers_time[stop_marker_index_list]

    # keep only the first start time for each stop time
    previous_stop_time = 0
    good_start_times = []
    for stop_time in stop_times_from_event_ide_tonic:
        possible_start_times = start_times_from_event_ide_tonic[
            start_times_from_event_ide_tonic < stop_time
        ]
        possible_start_times = possible_start_times[
            possible_start_times > previous_stop_time
        ]
        start_time = possible_start_times[0] if len(possible_start_times) > 0 else None
        good_start_times.append(start_time)
        previous_stop_time = stop_time
    good_start_times = np.array(good_start_times)
    good_start_times = good_start_times[
        good_start_times != None
    ]  # should be useless...

    good_start_times, stop_times_from_event_ide_tonic = get_coherent_start_stop_times(
        good_start_times,
        stop_times_from_event_ide_tonic,
    )

    return (
        good_start_times,
        stop_times_from_event_ide_tonic,
    )


def get_start_stop_times_from_mouse_markers(mouse_markers):
    """get the start and stop times from mouse_markers"""

    # NOTE: alternative way to get the start and stop times
    # here used to check the consistency with the mouse_to_nic_markers

    mouse_markers_data = mouse_markers["time_series"]
    mouse_markers_time = mouse_markers["time_stamps"]

    start_marker_index_list = find_marker_indexes(
        "DoCycleChange:DoRecord", mouse_markers_data
    )
    stop_marker_index_list = find_marker_indexes(
        "DoCycleChange:DoPause", mouse_markers_data
    )
    start_times_from_mouse_markers = mouse_markers_time[start_marker_index_list]
    stop_times_from_mouse_markers = mouse_markers_time[stop_marker_index_list]

    start_times_from_mouse_markers, stop_times_from_mouse_markers = (
        get_coherent_start_stop_times(
            start_times_from_mouse_markers,
            stop_times_from_mouse_markers,
        )
    )

    return (
        start_times_from_mouse_markers,
        stop_times_from_mouse_markers,
    )


def print_start_stop_times(start_stop_times):
    """Print the start and stop times from the (start_times, stop_times) tuple of lists"""

    start_times, stop_times = start_stop_times
    for i in range(len(start_times)):
        print(
            f"{i:02d}: {start_times[i]:8.2f} -> {stop_times[i]:8.2f}, Duration: {stop_times[i] - start_times[i]:5.2f}s"
        )

### Test the functions 

In [ ]:
## Test the functions

event_to_nic_markers = get_stream(xdf_data, "Markers", ["event_ide_TONIC"])
mouse_to_nic_markers = get_stream(xdf_data, "Markers", ["MouseToNIC"])

if mouse_to_nic_markers:
    mouse_to_nic_markers_data = mouse_to_nic_markers["time_series"]
    mouse_to_nic_markers_time = mouse_to_nic_markers["time_stamps"]
    start_t, stop_t = get_start_stop_times_from_mouse_to_nic_markers(
        mouse_to_nic_markers
    )
    print("Mouse to NIC markers:")
    print_start_stop_times((start_t, stop_t))
    # for the assert
    start_t_nic = start_t
    stop_t_nic = stop_t

if event_to_nic_markers:
    event_to_nic_markers_data = event_to_nic_markers["time_series"]
    event_to_nic_markers_time = event_to_nic_markers["time_stamps"]
    start_t, stop_t = get_start_stop_times_from_event_ide_TONIC(event_to_nic_markers)
    print("Event IDE to NIC markers:")
    print_start_stop_times((start_t, stop_t))

# NOTE: To verify that we get the same start and stop times from the mouse markers and the mouse to nic markers
mouse_markers = get_stream(xdf_data, "Markers", ["Mouse", "Mouse-Markers"])
if mouse_markers:
    mouse_markers_data = mouse_markers["time_series"]
    mouse_markers_time = mouse_markers["time_stamps"]
    start_t, stop_t = get_start_stop_times_from_mouse_markers(mouse_markers)

    # assert that the start and stop times are the same
    start_t_mouse = start_t
    stop_t_mouse = stop_t
    assert len(start_t_mouse) == len(start_t_nic)
    assert len(stop_t_mouse) == len(stop_t_nic)
    assert np.allclose(start_t_mouse, start_t_nic)
    assert np.allclose(stop_t_mouse, stop_t_nic)
    print("Mouse markers and NIC markers are coherent :-)")


if kinect_mocap:
    kinect_t = kinect_mocap["time_stamps"]
    WristRight_X = get_kinect_channel(kinect_mocap, "WristRight_X")
    WristRight_Y = get_kinect_channel(kinect_mocap, "WristRight_Y")
    WristRight_Z = get_kinect_channel(kinect_mocap, "WristRight_Z")

    WristLeft_X = get_kinect_channel(kinect_mocap, "WristLeft_X")
    WristLeft_Y = get_kinect_channel(kinect_mocap, "WristLeft_Y")
    WristLeft_Z = get_kinect_channel(kinect_mocap, "WristLeft_Z")

    WristLeft_Norm = np.sqrt(WristLeft_X**2 + WristLeft_Y**2 + WristLeft_Z**2)
    WristRight_Norm = np.sqrt(WristRight_X**2 + WristRight_Y**2 + WristRight_Z**2)
    print(f"kinect_mocap['time_series'] shape: {kinect_mocap['time_series'].shape}")


if event_to_nic_markers:
    event_markers_data = event_to_nic_markers["time_series"]
    event_markers_time = event_to_nic_markers["time_stamps"]
    print(f"Event markers data shape: {event_markers_data.shape}")
    print(f"Event markers time shape: {event_markers_time.shape}")

### Low pass filter the data
We choose a cutoff frequency of 0.5 Hz, which corresponds to a time constant of 2 seconds = typical time of a reach.

In [ ]:
def butter_lowpass(cutoff, fs, order=2):
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = butter(order, normal_cutoff, btype="low", analog=False)
    return b, a


def lowpass_filter(t, data):
    """Apply a lowpass filter to the data"""
    # check that the sampling period is constant
    dt = np.mean(np.diff(t))
    if not np.allclose(np.diff(t), dt):
        raise ValueError("The time vector do not have a constant sampling period")

    fs = 1 / dt  # sample rate, Hz
    cutoff = 0.5  # desired cutoff frequency of the filter, Hz
    order = 4  # order of the filter
    b, a = butter_lowpass(cutoff, fs, order=order)
    return filtfilt(b, a, data)

### Functions to identify the reaches
Includes debug options to visualize the data and the identified reaches.

In [ ]:
def index_of_last_negative_velocity_before_peak(t, velocity, t_end_i=None):
    """Get the index of the last negative velocity before the velocity peak"""

    if t_end_i is None:
        t_end_i = np.argmin(velocity)
    t_end = t[t_end_i]
    positive_before_t_end = velocity[t < t_end] > 0
    index_before_t_end = np.where(positive_before_t_end)[0]
    if len(index_before_t_end) == 0:
        return -1
    else:
        return max(index_before_t_end)


def get_one_reach(t, pos, t_end_i, do_plot=False):
    """
    get one reach from the position and time data
    """
    pos = np.array(pos)
    t = np.array(t)

    pos_f = lowpass_filter(t, pos)
    velocity = np.gradient(pos, t)
    f_velocity = lowpass_filter(t, velocity)

    # we know that the reach end
    t_end = t[t_end_i]
    # we look for the reach start
    t_beg_i = index_of_last_negative_velocity_before_peak(t, f_velocity, t_end_i)
    t_beg = t[t_beg_i]

    # verify that t_beg_i is not too close to the t_end_i
    if t_end_i - t_beg_i < 15:  # half a second
        t_beg_i = index_of_last_negative_velocity_before_peak(t, f_velocity, t_beg_i)

    # get the mask arround t_beg_*_i +/- 10
    t_beg_mask = range(t_beg_i - 10, t_beg_i + 10)
    t_end_mask = range(t_end_i - 10, t_end_i + 10)

    # we will return the median of the following positions
    beg_position = pos[t_beg_mask]
    end_position = pos[t_end_mask]

    # verify that the reach distance is larger than 0.05 m
    if np.abs(np.median(end_position) - np.median(beg_position)) < 0.05:
        return {
            "beg_position": np.nan,
            "end_position": np.nan,
            "t_beg": np.nan,
            "t_end": np.nan,
            "t_beg_i": np.nan,
            "t_end_i": np.nan,
        }

    if do_plot:

        def plot_one_sub(ax):
            ax.plot(t, pos, ".", label="Position", color="b")
            ax.plot(
                t[t_beg_mask],
                pos[t_beg_mask],
                "o",
                label="Beg reach t_beg_mask",
                color="g",
            )
            ax.plot(
                t[t_end_mask],
                pos[t_end_mask],
                "o",
                label="End reach t_end_mask",
                color="r",
            )

            ax.plot(
                t_beg,
                np.median(beg_position),
                "*",
                label="Beg reach median",
                color="g",
                markersize=20,
                markeredgewidth=2,
                markeredgecolor="k",
            )
            ax.plot(
                t_end,
                np.median(end_position),
                "*",
                label="End reach median",
                color="r",
                markersize=20,
                markeredgewidth=2,
                markeredgecolor="k",
            )

            ax.vlines(
                t_beg,
                ymin=np.min(pos),
                ymax=np.max(pos),
                color="g",
                linestyle="--",
                label="Beg reach",
            )
            ax.vlines(
                t_end,
                ymin=np.min(pos),
                ymax=np.max(pos),
                color="r",
                linestyle="--",
                label="End reach",
            )

            ax.hlines(
                np.median(beg_position),
                xmin=t_beg,
                xmax=t_end,
                color="g",
                linestyle="--",
            )
            ax.set_xlabel("Time (s)")
            ax.set_ylabel("Distance from the kinect (m)")

        # plot the data with 2 subplots
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 5))

        plot_one_sub(ax1)
        ax1.set_title("Whole reach series")
        plot_one_sub(ax2)
        t_zoom = t_beg - 2, t_end + 2
        ax2.set_xlim(t_zoom)
        # ax2.set_title("Zoom on the reach of interest")
        from matplotlib.patches import ConnectionPatch

        xy1 = (t_end, np.min(pos))
        xy2 = (t_end, np.max(pos))
        connection_end = ConnectionPatch(
            xyA=xy1,
            xyB=xy2,
            coordsA="data",
            coordsB="data",
            axesA=ax1,
            axesB=ax2,
            color="red",
        )
        ax2.add_artist(connection_end)

        xy1 = (t_beg, np.min(pos))
        xy2 = (t_beg, np.max(pos))
        connection_beg = ConnectionPatch(
            xyA=xy1,
            xyB=xy2,
            coordsA="data",
            coordsB="data",
            axesA=ax1,
            axesB=ax2,
            color="green",
        )
        ax2.add_artist(connection_beg)

        plt.show()

    return {
        "beg_position": np.median(beg_position),
        "end_position": np.median(end_position),
        "t_beg": t_beg,
        "t_end": t_end,
        "t_beg_i": t_beg_i,
        "t_end_i": t_end_i,
    }

In [ ]:
## get the reaches in the reaching time zone
def get_first_two_modes(x, bins=100):
    """Get the first two modes of the histogram"""
    # get the histogram of the data
    bin_heights, bin_edges = np.histogram(x, bins=bins)
    bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
    # get the peaks of the histogram
    peaks, _ = find_peaks(bin_heights, height=0)
    peak_heights = bin_heights[peaks]
    peak_centers = bin_centers[peaks]
    # get the two highest peaks
    sorted_peaks = np.argsort(peak_heights)[::-1]
    high_peaks = sorted_peaks[:2]
    high_peak_centers = peak_centers[high_peaks]

    return {
        "mode_1": high_peak_centers[0],
        "mode_2": high_peak_centers[1],
    }


def get_reaches(t, wrist, wrist_f):
    """get the reaches on this wrist"""
    wrist_center = (np.min(wrist) + np.max(wrist)) / 2
    modes = get_first_two_modes(wrist)
    inter_modes = 0.5 * (modes["mode_1"] + modes["mode_2"])

    threshold_median_iqr = np.median(wrist_f) - 3 * (
        np.percentile(wrist_f, 75) - np.percentile(wrist_f, 25)
    )

    find_peaks_threshold = np.median(wrist_f) - 0.1

    # find_peaks_threshold = threshold_median_iqr

    # find the negative peaks in the wrist that are below the threshold and at least 2 seconds apart
    inter_peaks_time = 60  # 2 seconds
    peaks, _ = find_peaks(
        -wrist_f, height=-find_peaks_threshold, distance=inter_peaks_time
    )

    if len(peaks) == 0:
        print("No peaks found")
        return None, None, None

    # remove the peaks that are outliers in the filtered data
    reaches_end = wrist_f[peaks]
    reaches_end_median = np.median(reaches_end)
    reaches_end_iqr = np.percentile(reaches_end, 75) - np.percentile(reaches_end, 25)
    i_outliers = [
        i
        for i in range(len(reaches_end))
        if abs(reaches_end[i] - reaches_end_median) > 3 * reaches_end_iqr
    ]
    peaks = np.delete(peaks, i_outliers)

    # get the reaches (t_beg, t_end)
    reaches = []
    for pk in peaks:
        reach = get_one_reach(t, wrist, t_end_i=pk, do_plot=False)
        reaches.append(reach)

    return peaks, reaches, find_peaks_threshold


def restrict_to_reach_zone(t, x):
    out = x[(t >= reaching_time_zone["start"]) & (t <= reaching_time_zone["stop"])]
    return np.array(out)


reach_left_f = lowpass_filter(reach_t, reach_left)
reach_right_f = lowpass_filter(reach_t, reach_right)

peaks_left, reaches_left, thresh_left = get_reaches(reach_t, reach_left, reach_left_f)
peaks_right, reaches_right, thresh_right = get_reaches(
    reach_t, reach_right, reach_right_f
)

if do_visualize:

    def plot_reaches(
        ax, t, wrist, wrist_f, i_peaks, reaches, label="wrist", color="b", thresh=None
    ):
        """ " Plot the reaches of the wrist"""
        ax.plot(t, wrist, ".", label=label, color=color)
        ax.plot(t, wrist_f, label=f"{label} filtered", color=color, alpha=0.2)
        ax.plot(
            t[i_peaks],
            wrist_f[i_peaks],
            "x",
            color="r",
        )
        # plot the threshold
        if thresh is not None:
            ax.axhline(
                y=thresh,
                color=color,
                linestyle="--",
                label=f"Threshold {label}",
            )
        for reach in reaches:
            ax.plot(
                reach["t_beg"],
                reach["beg_position"],
                "o",
                color="orange",
            )
            ax.plot(
                reach["t_end"],
                reach["end_position"],
                "o",
                color="r",
            )
            ax.plot(
                [reach["t_beg"], reach["t_end"]],
                [reach["beg_position"], reach["end_position"]],
                color="k",
                linestyle="--",
            )

        ax.set_xlabel("Time (s)")
        ax.set_ylabel("Distance from the kinect (m)")
        ax.legend()

    # plot the signal and the peaks
    fig, ax = plt.subplots(figsize=(10, 5))
    if peaks_left is not None:
        plot_reaches(
            ax,
            reach_t,
            reach_left,
            reach_left_f,
            peaks_left,
            reaches_left,
            label="Left wrist",
            color="b",
            thresh=thresh_left,
        )
    if peaks_right is not None:
        plot_reaches(
            ax,
            reach_t,
            reach_right,
            reach_right_f,
            peaks_right,
            reaches_right,
            label="Right wrist",
            color="k",
            thresh=thresh_right,
        )

    plt.show()